In [0]:
%run "./00_Pipeline_Configurations"

In [0]:
%run "./08_Audit_Log_Function"

In [0]:
# 02_Silver_Layer_Students

from datetime import datetime

start_time = datetime.now()

try:
    df_students = spark.table(bronze_students_table)
    rows_before = df_students.count()

    df_students_clean = (
        df_students
        .dropDuplicates(["student_id"])
        .dropna()   # drops null
    )

    rows_after = df_students_clean.count()

    df_students_clean.write.format("delta").mode("overwrite").saveAsTable(silver_students_table)

    display(df_students_clean)

    print(f"Rows before cleaning: {rows_before}")
    print(f"Rows after cleaning: {rows_after}")
    print(f"Rows dropped (nulls/duplicates): {rows_before - rows_after}")

    log_audit("silver_students", table_name=silver_students_table, status="SUCCESS", start_time=start_time)

except Exception as e:
    log_audit("silver_students", status="FAILED", start_time=start_time, error_msg=str(e))
    raise